In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error as mse
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

In [ ]:
training_data = pd.read_csv('training_data_case1.csv')
X = training_data[['F', 'Cai', 'Fc', 'Cac']]
y = training_data[['Kp', 'tauI', 'tauD']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 500, random_state = 8)
print(y_train)

### For Case 2 - 10,000 training data points ###
# training_data = pd.read_csv('training_data_case2-10k.csv')
# testing_data = pd.read_csv('validation_data_case2.csv')
# X_train = training_data[['tau1', 'tau2', 'Cac/F']]
# y_train = training_data[['Kp', 'tauI', 'tauD']]
# X_test = testing_data[['tau1', 'tau2', 'Cac/F']]
# y_test = testing_data[['Kp', 'tauI', 'tauD']]

### For Case 2 - 100,000 training data points ###
# training_data = pd.read_csv('training_data_case2-100k.csv')
# testing_data = pd.read_csv('validation_data_case2.csv')
# X_train = training_data[['tau1', 'tau2', 'Cac/F']]
# y_train = training_data[['Kp', 'tauI', 'tauD']]
# X_test = testing_data[['tau1', 'tau2', 'Cac/F']]
# y_test = testing_data[['Kp', 'tauI', 'tauD']]

In [ ]:
# Number of 100-epoch training cycles for each architecture.
# Section 5.2.3: Case 1 NNs are trained for 600 epochs (6 cycles), Case 2 NNs for 1000 epochs (10 cycles).
n_cycles = 6

bestr2 = 0.5
bestrmse = 2

f2 = open('case1_nn_hyperparameter_search.csv','w')
f2.write('layer1,layer2,n_epochs,kpr2,tauIr2,tauDr2,overallr2,kpmse,tauImse,tauDmse,overallmse,kprmse,tauIrmse,tauDrmse,overallrmse\n')
f2.close()
for i in range(10, 16):
    for j in range(8, 16):
        if j <= i:
            number_of_features = 4
            n_outputs = 3
            model = Sequential()
            model.add(Input(shape = (number_of_features,))) #Input Layer where the shape is the number of features
            model.add(Dense(i, activation='relu')) #Hidden Layer 1
            model.add(Dense(j, activation='relu')) #Hidden Layer 2
            model.add(Dense(n_outputs, activation='linear')) # Output layer
            opt = Adam(learning_rate=0.002) #Optimizer - how it takes the current weights and determines the next weights
            model.compile(loss='mse', optimizer=opt, metrics=['mean_squared_error'])

            # Train in 100-epoch cycles; the NN is saved and evaluated after every cycle
            for cycle in range(1, n_cycles + 1):
                n_epochs = 100 * cycle # Total epochs trained so far

                model.fit(X_train, y_train, epochs = 100, batch_size = 128, verbose = 1)
                y_predicted = model.predict(X_test)
                newr2 = r2_score(y_test, y_predicted)
                newrmse = np.sqrt(mse(y_test, y_predicted))

                if newrmse < bestrmse:
                    nn_file_name = 'nns_case1/' + str(i) + '_' + str(j) + '_' + str(n_epochs) + 'x'
                    model.save(nn_file_name)
                    bestrmse = newrmse
                    if newr2 > bestr2:
                        bestr2 = newr2
                elif newr2 > bestr2:
                    nn_file_name = 'nns_case1/' + str(i) + '_' + str(j) + '_' + str(n_epochs) + 'x'
                    model.save(nn_file_name)
                    bestr2 = newr2
                    if newrmse > bestrmse:
                        bestrmse = newrmse

                str1 = str(r2_score(y_test.Kp, y_predicted[:,0]))
                str2 = str(r2_score(y_test.tauI, y_predicted[:,1]))
                str3 = str(r2_score(y_test.tauD, y_predicted[:,2]))
                str4 = str(r2_score(y_test, y_predicted))
                str5 = str(mse(y_test.Kp, y_predicted[:,0]))
                str6 = str(mse(y_test.tauI, y_predicted[:,1]))
                str7 = str(mse(y_test.tauD, y_predicted[:,2]))
                str8 = str(mse(y_test, y_predicted))
                str9 = str(np.sqrt(mse(y_test.Kp, y_predicted[:,0])))
                str10 = str(np.sqrt(mse(y_test.tauI, y_predicted[:,1])))
                str11 = str(np.sqrt(mse(y_test.tauD, y_predicted[:,2])))
                str12 = str(np.sqrt(mse(y_test, y_predicted)))

                string2 = str(i) + ',' + str(j) + ',' + str(n_epochs) + ',' + str1 + ',' + str2 + ',' + str3 + ',' + str4 + ',' + str5 + ',' + str6 + ',' + str7 + ',' + str8 + ',' + str9 + ',' + str10 + ',' + str11 + ',' + str12
                f2 = open('case1_nn_hyperparameter_search.csv','a')
                f2.write(string2 + '\n')
                f2.close()

In [ ]:
# ### For Case 2, 10,000 data points ###
# n_cycles = 6

# bestr2 = 0.5
# bestrmse = 2

# f2 = open('case2-10k_nn_hyperparameter_search.csv','w')
# f2.write('layer1,layer2,n_epochs,kpr2,tauIr2,tauDr2,overallr2,kpmse,tauImse,tauDmse,overallmse,kprmse,tauIrmse,tauDrmse,overallrmse\n')
# f2.close()
# for i in range(10, 16):
#     for j in range(8, 16):
#         if j <= i:
#             number_of_features = 3
#             n_outputs = 3
#             model = Sequential()
#             model.add(Input(shape = (number_of_features,))) #Input Layer where the shape is the number of features
#             model.add(Dense(i, activation='relu')) #Hidden Layer 1
#             model.add(Dense(j, activation='relu')) #Hidden Layer 2
#             model.add(Dense(n_outputs, activation='linear')) # Output layer
#             opt = Adam(learning_rate=0.002) #Optimizer - how it takes the current weights and determines the next weights
#             model.compile(loss='mse', optimizer=opt, metrics=['mean_squared_error'])

#             # Train in 100-epoch cycles; the NN is saved and evaluated after every cycle
#             for cycle in range(1, n_cycles + 1):
#                 n_epochs = 100 * cycle # Total epochs trained so far

#                 model.fit(X_train, y_train, epochs = 100, batch_size = 128, verbose = 1)
#                 y_predicted = model.predict(X_test)
#                 newr2 = r2_score(y_test, y_predicted)
#                 newrmse = np.sqrt(mse(y_test, y_predicted))

#                 if newrmse < bestrmse:
#                     nn_file_name = 'nns_case2-10k/' + str(i) + '_' + str(j) + '_' + str(n_epochs) + 'x'
#                     model.save(nn_file_name)
#                     bestrmse = newrmse
#                     if newr2 > bestr2:
#                         bestr2 = newr2
#                 elif newr2 > bestr2:
#                     nn_file_name = 'nns_case2-10k/' + str(i) + '_' + str(j) + '_' + str(n_epochs) + 'x'
#                     model.save(nn_file_name)
#                     bestr2 = newr2
#                     if newrmse > bestrmse:
#                         bestrmse = newrmse

#                 str1 = str(r2_score(y_test.Kp, y_predicted[:,0]))
#                 str2 = str(r2_score(y_test.tauI, y_predicted[:,1]))
#                 str3 = str(r2_score(y_test.tauD, y_predicted[:,2]))
#                 str4 = str(r2_score(y_test, y_predicted))
#                 str5 = str(mse(y_test.Kp, y_predicted[:,0]))
#                 str6 = str(mse(y_test.tauI, y_predicted[:,1]))
#                 str7 = str(mse(y_test.tauD, y_predicted[:,2]))
#                 str8 = str(mse(y_test, y_predicted))
#                 str9 = str(np.sqrt(mse(y_test.Kp, y_predicted[:,0])))
#                 str10 = str(np.sqrt(mse(y_test.tauI, y_predicted[:,1])))
#                 str11 = str(np.sqrt(mse(y_test.tauD, y_predicted[:,2])))
#                 str12 = str(np.sqrt(mse(y_test, y_predicted)))

#                 string2 = str(i) + ',' + str(j) + ',' + str(n_epochs) + ',' + str1 + ',' + str2 + ',' + str3 + ',' + str4 + ',' + str5 + ',' + str6 + ',' + str7 + ',' + str8 + ',' + str9 + ',' + str10 + ',' + str11 + ',' + str12
#                 f2 = open('case2-10k_nn_hyperparameter_search.csv','a')
#                 f2.write(string2 + '\n')
#                 f2.close()

In [ ]:
# ### For Case 2, 100,000 data points ###
# n_cycles = 6

# bestr2 = 0.5
# bestrmse = 2

# f2 = open('case2-100k_nn_hyperparameter_search.csv','w')
# f2.write('layer1,layer2,n_epochs,kpr2,tauIr2,tauDr2,overallr2,kpmse,tauImse,tauDmse,overallmse,kprmse,tauIrmse,tauDrmse,overallrmse\n')
# f2.close()
# for i in range(10, 16):
#     for j in range(8, 16):
#         if j <= i:
#             number_of_features = 3
#             n_outputs = 3
#             model = Sequential()
#             model.add(Input(shape = (number_of_features,))) #Input Layer where the shape is the number of features
#             model.add(Dense(i, activation='relu')) #Hidden Layer 1
#             model.add(Dense(j, activation='relu')) #Hidden Layer 2
#             model.add(Dense(n_outputs, activation='linear')) # Output layer
#             opt = Adam(learning_rate=0.002) #Optimizer - how it takes the current weights and determines the next weights
#             model.compile(loss='mse', optimizer=opt, metrics=['mean_squared_error'])

#             # Train in 100-epoch cycles; the NN is saved and evaluated after every cycle
#             for cycle in range(1, n_cycles + 1):
#                 n_epochs = 100 * cycle # Total epochs trained so far

#                 model.fit(X_train, y_train, epochs = 100, batch_size = 128, verbose = 1)
#                 y_predicted = model.predict(X_test)
#                 newr2 = r2_score(y_test, y_predicted)
#                 newrmse = np.sqrt(mse(y_test, y_predicted))

#                 if newrmse < bestrmse:
#                     nn_file_name = 'nns_case2-100k/' + str(i) + '_' + str(j) + '_' + str(n_epochs) + 'x'
#                     model.save(nn_file_name)
#                     bestrmse = newrmse
#                     if newr2 > bestr2:
#                         bestr2 = newr2
#                 elif newr2 > bestr2:
#                     nn_file_name = 'nns_case2-100k/' + str(i) + '_' + str(j) + '_' + str(n_epochs) + 'x'
#                     model.save(nn_file_name)
#                     bestr2 = newr2
#                     if newrmse > bestrmse:
#                         bestrmse = newrmse


#                 str1 = str(r2_score(y_test.Kp, y_predicted[:,0]))
#                 str2 = str(r2_score(y_test.tauI, y_predicted[:,1]))
#                 str3 = str(r2_score(y_test.tauD, y_predicted[:,2]))
#                 str4 = str(r2_score(y_test, y_predicted))
#                 str5 = str(mse(y_test.Kp, y_predicted[:,0]))
#                 str6 = str(mse(y_test.tauI, y_predicted[:,1]))
#                 str7 = str(mse(y_test.tauD, y_predicted[:,2]))
#                 str8 = str(mse(y_test, y_predicted))
#                 str9 = str(np.sqrt(mse(y_test.Kp, y_predicted[:,0])))
#                 str10 = str(np.sqrt(mse(y_test.tauI, y_predicted[:,1])))
#                 str11 = str(np.sqrt(mse(y_test.tauD, y_predicted[:,2])))
#                 str12 = str(np.sqrt(mse(y_test, y_predicted)))

#                 string2 = str(i) + ',' + str(j) + ',' + str(n_epochs) + ',' + str1 + ',' + str2 + ',' + str3 + ',' + str4 + ',' + str5 + ',' + str6 + ',' + str7 + ',' + str8 + ',' + str9 + ',' + str10 + ',' + str11 + ',' + str12
#                 f2 = open('case2-100k_nn_hyperparameter_search.csv','a')
#                 f2.write(string2 + '\n')
#                 f2.close()

In [ ]:
# Some statistics
print("Minimums:")
print("Training data", y_train[['Kp']].min)
print("Training data", y_train[['tauI']].min)
print("Training data", y_train[['tauD']].min)
print("Testing data", y_train[['Kp']].min)
print("Testing data", y_train[['tauI']].min)
print("Testing data", y_train[['tauD']].min)

print("Maximums:")
print("Training data", y_train[['Kp']].max())
print("Training data", y_train[['tauI']].max())
print("Training data", y_train[['tauD']].max())
print("Testing data", y_train[['Kp']].max())
print("Testing data", y_train[['tauI']].max())
print("Testing data", y_train[['tauD']].max())

print("Number of unique values:")
print("Training data", y_train[['Kp']].nunique())
print("Training data", y_train[['tauI']].nunique())
print("Training data", y_train[['tauD']].nunique())
print("Testing data", y_train[['Kp']].nunique())
print("Testing data", y_train[['tauI']].nunique())
print("Testing data", y_train[['tauD']].nunique())

In [ ]:
# Evaluate NN performance - change file name as desired
datadf = pd.read_csv("case1_nn_hyperparameter_search.csv")
# datadf = pd.read_csv("case2-10k_nn_hyperparameter_search.csv")
# datadf = pd.read_csv("case2-100k_nn_hyperparameter_search.csv")
print(datadf.nsmallest(3, 'kprmse'))
print(datadf.nsmallest(3, 'tauIrmse'))
print(datadf.nsmallest(3, 'tauDrmse'))
print(datadf.nsmallest(3, 'overallrmse'))
print(datadf.nlargest(3, 'kpr2'))
print(datadf.nlargest(3, 'tauIr2'))
print(datadf.nlargest(3, 'tauDr2'))
print(datadf.nlargest(3, 'overallr2'))

In [2]:
# Example:
datadf = pd.read_csv("old_case1_nn_hyperparameter_search.csv")
print(datadf.nsmallest(3, 'kprmse')[['layer1', 'layer2', 'n_epochs', 'kprmse']])
print(datadf.nsmallest(3, 'tauIrmse')[['layer1', 'layer2', 'n_epochs', 'tauIrmse']])
print(datadf.nsmallest(3, 'tauDrmse')[['layer1', 'layer2', 'n_epochs', 'tauDrmse']])
print(datadf.nsmallest(3, 'overallrmse')[['layer1', 'layer2', 'n_epochs', 'overallrmse']])
print(datadf.nlargest(3, 'kpr2')[['layer1', 'layer2', 'n_epochs', 'kpr2']])
print(datadf.nlargest(3, 'tauIr2')[['layer1', 'layer2', 'n_epochs', 'tauIr2']])
print(datadf.nlargest(3, 'tauDr2')[['layer1', 'layer2', 'n_epochs', 'tauDr2']])
print(datadf.nlargest(3, 'overallr2')[['layer1', 'layer2', 'n_epochs', 'overallr2']])

    layer1  layer2  n_epochs    kprmse
7       14      12       600  0.579939
3       13      12       600  0.581324
10      14      14       600  0.598342
   layer1  layer2  n_epochs  tauIrmse
1      13      10       400  0.779984
0      13      10       300  0.798032
8      14      13       500  0.820105
   layer1  layer2  n_epochs  tauDrmse
5      14      12       300  0.018254
0      13      10       300  0.018434
1      13      10       400  0.019368
   layer1  layer2  n_epochs  overallrmse
1      13      10       400     0.602555
0      13      10       300     0.606722
9      14      14       400     0.660380
    layer1  layer2  n_epochs      kpr2
9       14      14       400  0.998791
10      14      14       600  0.998710
4       14       8       600  0.998510
    layer1  layer2  n_epochs    tauIr2
2       13      12       400  0.994408
3       13      12       600  0.993872
11      15      12       400  0.992903
    layer1  layer2  n_epochs    tauDr2
12      15      12       